# triangle-barycentric — worked example 3: Distance from a point to the nearest triangle edge in barycentric space

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `triangle-barycentric`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

In barycentric space the three edges of the triangle correspond to the constraints `u = 0`, `v = 0`, and `u + v = 1`. The distance from an interior or exterior point `(u, v)` to the nearest edge is the minimum of `|u|`, `|v|`, and `|1 - u - v|` (unsigned). Points near an edge have a small minimum distance; the centroid `(1/3, 1/3)` maximizes it.

## Worked solution

**Step 1 – Compute the three edge distances.** For each point `(u, v)` in barycentric coordinates: `d0 = u.abs()` (distance to edge u=0), `d1 = v.abs()` (distance to edge v=0), `d2 = (1 - u - v).abs()` (distance to edge u+v=1).

**Step 2 – Minimum over the three.** `t.stack([d0, d1, d2], dim=1).min(dim=1).values` gives the minimum edge distance for each point.

**Step 3 – Centroid maximizes it.** The centroid `(1/3, 1/3)` has `d0 = d1 = 1/3` and `d2 = |1 - 2/3| = 1/3`. All three distances equal 1/3, which is the maximum achievable for the unit simplex.

**Step 4 – Return.** We return a dict with the distances per point and the argmax (the point furthest from any edge).

In [ ]:
import torch as t

def worked3_edge_distances(uvs):
    """
    Compute each point's minimum distance to the three edges of the unit simplex.
    Returns dict with per-point min-edge distances and the index of the most interior point.
    """
    u = uvs[:, 0]
    v = uvs[:, 1]
    d0 = u.abs()              # distance to u = 0 edge
    d1 = v.abs()              # distance to v = 0 edge
    d2 = (1.0 - u - v).abs() # distance to u + v = 1 edge
    distances = t.stack([d0, d1, d2], dim=1)  # (N, 3)
    min_dist = distances.min(dim=1).values    # (N,)
    most_interior_idx = min_dist.argmax().item()
    return {
        'min_edge_dist': min_dist,
        'most_interior_idx': most_interior_idx,
    }

# Demo with centroid and several other points
t.manual_seed(0)
test_uvs = t.tensor([
    [1/3, 1/3],   # centroid — furthest from all edges
    [0.05, 0.05], # near corner A
    [0.9, 0.05],  # near edge u+v=1
    [0.5, 0.5],   # exactly on edge u+v=1
])
result = worked3_edge_distances(test_uvs)
print('min edge distances:', result['min_edge_dist'])
print('most interior point idx:', result['most_interior_idx'], '(should be 0 = centroid)')